# DPO

In [2]:
!pip install transformers accelerate hf-transfer peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 35.6 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

In [3]:
model_id = 'LGAI-EXAONE/EXAONE-4.0-1.2B'

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.bfloat16,
    device_map='auto'
)

tokenizer = AutoTokenizer.from_pretrained(model_id)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.56G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/332 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

In [16]:
# 샘플 데이터
prompt = '파이썬이 뭐야?'
chosen = '파이썬은 배우기 쉽고, 강력한 프로그래밍 언어입니다.'
rejected = '파이썬은 뱀이지~'

In [18]:
# 생성 확률 계산

def get_logprob(model, tokenizer, prompt, response):
  """주어진 prompt 뒤에 response가 이어질 로그 확률을 계산하는 함수"""

  # 모델은 이 전체 문장을 보고 각 위치의 다음 토큰 확률을 계산
  full_text = prompt + response

  # 전체 문장 토큰화
  inputs = tokenizer(full_text, return_tensors='pt').to(model.device)

  # prompt가 몇 개의 토큰으로 구성 되어 있는지 계산
  prompt_len = len(tokenizer(prompt, return_tensors='pt')['input_ids'][0])

  with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits

    # logits을 log 확률로 변환
    log_probs = F.log_softmax(logits, dim=-1)

    # print(log_probs.shape)
    # 마지막 위치의 예측은 실제 다음 토큰이 없으므로 제외
    # print(log_probs[:, :-1].shape)
    # 첫 토큰을 제외하고 각 위치에서 모델이 맞혀야할 실제 다음 토큰
    # print(inputs['input_ids'][:, 1:].shape)

    # n번째 위치의 logits가 n+1번째 위치의 토큰을 예측한다.
    # gather: vacab 전체 확률 중 실제 정답 토큰 ID에 해당하는 로그 확률만 뽑는다.
    token_log_probs = log_probs[:, :-1].gather(
        index=inputs['input_ids'][:, 1:].unsqueeze(-1),
        dim=-1
    ).squeeze(-1)

    # response의 첫 토큰 기준으로 분리
    response_log_probs = token_log_probs[:, prompt_len - 1:]

    # response를 구성하는 모든 토큰의 로그 확률을 더한다.
    # 로그 확률의 합은 response 전체 문장의 로그 확률로 볼 수 있다.
    total = response_log_probs.sum()

  return total.item()

chosen_prob = get_logprob(model, tokenizer, prompt, chosen)
rejected_prob = get_logprob(model, tokenizer, prompt, rejected)
print(f'선택 된 답변 생성 확률 : {chosen_prob}')
print(f'거절 된 답변 생성 확률 : {rejected_prob}')


선택 된 답변 생성 확률 : -85.0
거절 된 답변 생성 확률 : -59.0


In [ ]:
# DPO 손실 함수
# - chosen 답변은 더 높은 확률을 가지도록 하고,
#   rejected 답변은 더 낮은 확률을 갖도록 학습한다.
# policy model(정책 모델) / reference model(참조 모델)

def dpo_loss(policy_chosen, policy_rejected, ref_chosen, ref_rejected, beta=0.1):

  # policy_chosen - policy_rejected
  # 현재 학습 대상 모델이 chosen을 rejected보다 얼마나 더 선호하는지를 나타낸다.

  # ref_chosen - ref_rejected
  # 기준이 되는 참조 모델이 chosen을 rejected보다 얼마나 더 선호하는지 나타낸다.

  # 정책 모델의 선호차이가 참조 모델의 선호차이보다 커지도록 학습
  logits = beta * ((policy_chosen - policy_rejected) - (ref_chosen - ref_rejected))

  # logsigmoid: logits 값이 클수록 손실이 작아진다.
  # => 정책 모델이 chosen을 더 선호하면 loss가 작아진다.
  loss = -F.logsigmoid(torch.tensor(logits))

  return loss.item()
